# 02 · Build canonical 72 h master prediction table

Builds the validated 71,010-row non-training prediction corpus with provenance and leakage checks.

**Checkpoint date:** 2026-10-05.

In [ ]:
#!/usr/bin/env python3
"""
Build the canonical Gray-Box AIA72 master prediction table for all NON-TRAIN roles.

Safety:
- no GPU use
- no model fitting
- no deletion
- no GCS mutation
- refuses to overwrite output
- validates every case ID against the frozen schedule
- validates the manually recovered supplementary block against its proof archive
"""

from pathlib import Path
import hashlib
import io
import json
import tarfile
import pandas as pd

OLD = Path("/home/abmoses2000/graybox_aia72_inference_v1_20261004")
OLD_PRED = OLD / "execution" / "predictions"
CONT = Path("/home/abmoses2000/graybox_aia72_inference_cont2_20261005")
CONT_PRED = CONT / "predictions"

CASES = OLD / "inputs" / "cases.csv.gz"
BLOCKS = OLD / "inputs" / "blocks.json"
ACCEPTED = OLD_PRED / "accepted_selection_predictions.csv.gz"

REC_BLOCK = "supplementary_2026_002048_002560"
REC_ARCHIVE = OLD_PRED / "proofs" / f"{REC_BLOCK}.tar.gz"
REC_ARCHIVE_SHA = "b32e976538b8637354ec4d28b8409e99c107a579b878c4ff85df51bd1f5fba20"
REC_PRED_SHA = "176acea751a24d07c7806b79d07e71b4fc108e066c3127fb1ccd276344358ff1"

OUT = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

MASTER = OUT / "graybox_aia72_master_predictions.csv.gz"
MANIFEST = OUT / "manifest.json"
SOURCE_TABLE = OUT / "source_artifacts.csv"

EXPECTED_TOTAL = 71010
EXPECTED_BY_ROLE = {
    "model_validation": 3905,
    "probability_calibration": 2831,
    "conformal_calibration": 4168,
    "policy_validation": 13142,
    "retrospective_cycle25": 35846,
    "supplementary_2026": 11118,
}
BLOCK_ROLES = {
    "probability_calibration",
    "conformal_calibration",
    "policy_validation",
    "retrospective_cycle25",
    "supplementary_2026",
}

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond, msg):
    if not cond:
        raise ValueError(msg)

def read_pred(path):
    df = pd.read_csv(path, low_memory=False)
    require("forecast_case_id" in df.columns, f"Missing forecast_case_id: {path}")
    require(df["forecast_case_id"].is_unique, f"Duplicate forecast_case_id within {path}")
    return df

cases = pd.read_csv(CASES, low_memory=False)
blocks = json.loads(BLOCKS.read_text())

require(len(cases) == 96596, "Frozen case count changed")
require(cases["forecast_case_id"].is_unique, "Frozen case IDs are not unique")

role_cases = {
    role: g.reset_index(drop=True)
    for role, g in cases.groupby("role", sort=False)
}

for role, n in EXPECTED_BY_ROLE.items():
    require(role in role_cases, f"Missing role in frozen cases: {role}")
    require(len(role_cases[role]) == n, f"Frozen role count changed: {role}")

# The 3,905 model-validation cases are intentionally reused from accepted selection.
require(ACCEPTED.is_file(), "accepted_selection_predictions.csv.gz missing")
accepted = read_pred(ACCEPTED)
expected_mv = role_cases["model_validation"]["forecast_case_id"].astype(str).tolist()
actual_mv = accepted["forecast_case_id"].astype(str).tolist()
require(len(accepted) == 3905, "Accepted-selection row count changed")
require(actual_mv == expected_mv, "Accepted-selection IDs/order do not exactly match model_validation")

accepted = accepted.copy()
accepted["_prediction_source"] = "accepted_selection_reuse"
accepted["_prediction_block_id"] = "accepted_selection"

source_rows = [{
    "role": "model_validation",
    "block_id": "accepted_selection",
    "source": "accepted_selection_reuse",
    "path": str(ACCEPTED),
    "rows": len(accepted),
    "sha256": sha256(ACCEPTED),
}]

# Verify the recovered proof archive and confirm the local prediction file is byte-identical.
require(REC_ARCHIVE.is_file(), "Recovered proof archive missing")
require(sha256(REC_ARCHIVE) == REC_ARCHIVE_SHA, "Recovered proof archive SHA changed")

with tarfile.open(REC_ARCHIVE, "r:gz") as tar:
    pred_name = f"{REC_BLOCK}_predictions.csv.gz"
    require(pred_name in tar.getnames(), "Recovered prediction member missing")
    recovered_raw = tar.extractfile(pred_name).read()
    require(hashlib.sha256(recovered_raw).hexdigest() == REC_PRED_SHA,
            "Recovered prediction member SHA changed")

# Build all scheduled non-train/non-model-validation blocks.
frames = [accepted]
seen_ids = set(actual_mv)
scheduled_blocks = 0

for b in blocks:
    role = b["role"]
    if role not in BLOCK_ROLES:
        continue

    scheduled_blocks += 1
    bid = b["block_id"]

    p_old = OLD_PRED / f"{bid}_predictions.csv.gz"
    p_cont = CONT_PRED / f"{bid}_predictions.csv.gz"

    exists = [p for p in (p_old, p_cont) if p.is_file()]
    require(len(exists) == 1, f"Expected exactly one canonical local prediction file for {bid}; found {exists}")
    p = exists[0]
    source = "original" if p == p_old else "continuation_2"

    if bid == REC_BLOCK:
        require(sha256(p) == REC_PRED_SHA,
                "Local recovered-block prediction does not match verified proof archive")

    df = read_pred(p)
    expected_ids = list(map(str, b["case_ids"]))
    actual_ids = df["forecast_case_id"].astype(str).tolist()

    require(len(df) == len(expected_ids), f"Row count mismatch for {bid}")
    require(actual_ids == expected_ids, f"Case IDs/order mismatch for {bid}")

    overlap = seen_ids.intersection(actual_ids)
    require(not overlap, f"Cross-block duplicate case IDs in {bid}: {len(overlap)}")
    seen_ids.update(actual_ids)

    df = df.copy()
    df["_prediction_source"] = source
    df["_prediction_block_id"] = bid
    frames.append(df)

    source_rows.append({
        "role": role,
        "block_id": bid,
        "source": source,
        "path": str(p),
        "rows": len(df),
        "sha256": sha256(p),
    })

require(scheduled_blocks == 134, f"Expected 134 inference blocks, got {scheduled_blocks}")

master = pd.concat(frames, ignore_index=True, sort=False)
require(len(master) == EXPECTED_TOTAL, f"Master row count mismatch: {len(master)}")
require(master["forecast_case_id"].is_unique, "Master case IDs are not unique")

# Every non-train frozen case must appear exactly once; no train case may appear.
expected_nontrain = cases.loc[cases["role"] != "train", "forecast_case_id"].astype(str).tolist()
actual_ids = master["forecast_case_id"].astype(str).tolist()
require(set(actual_ids) == set(expected_nontrain), "Master ID set does not equal all frozen non-train cases")

train_ids = set(cases.loc[cases["role"] == "train", "forecast_case_id"].astype(str))
require(not train_ids.intersection(actual_ids), "Training cases leaked into master prediction table")

# Reorder master exactly to the frozen non-train case order.
order = {cid: i for i, cid in enumerate(expected_nontrain)}
master["_frozen_case_order"] = master["forecast_case_id"].astype(str).map(order)
require(master["_frozen_case_order"].notna().all(), "Failed to map frozen case order")
master = master.sort_values("_frozen_case_order").reset_index(drop=True)

# Verify roles against the authoritative frozen case table.
# Some reused prediction artefacts (notably accepted_selection_predictions.csv.gz)
# legitimately contain no role column. After concatenation that creates NaN values,
# so compare only non-null source roles, preserve them for provenance, and then set
# the frozen role as the canonical role used by the master table.
authoritative = cases[["forecast_case_id", "role"]].copy()
authoritative["forecast_case_id"] = authoritative["forecast_case_id"].astype(str)
master["forecast_case_id"] = master["forecast_case_id"].astype(str)

role_map = authoritative.set_index("forecast_case_id")["role"]
expected_role = master["forecast_case_id"].map(role_map)
require(expected_role.notna().all(), "A master case is missing from the frozen role map")

if "role" in master.columns:
    master["_source_role"] = master["role"]
    present = master["_source_role"].notna()
    require(
        (master.loc[present, "_source_role"].astype(str)
         == expected_role.loc[present].astype(str)).all(),
        "A non-null prediction role disagrees with the frozen role"
    )
else:
    master["_source_role"] = pd.NA

master["role"] = expected_role.astype(str)

counts = master.groupby("role", sort=False).size().to_dict()
require(counts == EXPECTED_BY_ROLE, f"Master role counts changed: {counts}")

# Deterministic gzip output.
master.to_csv(
    MASTER,
    index=False,
    compression={"method": "gzip", "mtime": 0},
)

sources = pd.DataFrame(source_rows)
sources.to_csv(SOURCE_TABLE, index=False)

manifest = {
    "status": "MASTER_PREDICTION_TABLE_BUILT_AND_VALIDATED",
    "scientific_acceptance": False,
    "purpose": "Canonical 72-hour Gray-Box prediction corpus before calibration/UQ/policy analysis",
    "frozen_cases_sha256": sha256(CASES),
    "frozen_blocks_sha256": sha256(BLOCKS),
    "master_path": str(MASTER),
    "master_sha256": sha256(MASTER),
    "master_rows": int(len(master)),
    "master_unique_case_ids": int(master["forecast_case_id"].nunique()),
    "roles": {k: int(v) for k, v in counts.items()},
    "training_rows_excluded": int((cases["role"] == "train").sum()),
    "model_validation_rows_reused_from_accepted_selection": 3905,
    "scheduled_inference_blocks_included": scheduled_blocks,
    "prediction_source_counts": {
        k: int(v) for k, v in sources.groupby("source")["rows"].sum().to_dict().items()
    },
    "recovered_block": {
        "block_id": REC_BLOCK,
        "proof_archive_sha256": REC_ARCHIVE_SHA,
        "prediction_sha256": REC_PRED_SHA,
        "local_prediction_matches_proof": True,
    },
    "no_duplicate_case_ids": True,
    "no_training_case_leakage": True,
    "all_nontrain_frozen_case_ids_present": True,
    "note": "scientific_acceptance remains false until downstream protocol stages are completed and reviewed.",
}
MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n")

print("===== MASTER TABLE COMPLETE =====")
print(json.dumps(manifest, indent=2))
print("\nSaved:", MASTER)
print("Saved:", MANIFEST)
print("Saved:", SOURCE_TABLE)
